In [1]:
import duckdb
import numpy as np
import pandas as pd


In [2]:
df=duckdb.sql("SELECT * FROM read_parquet('..\data\model\model_dataset.parquet')").df()
train=df[df["split"]=="train"].copy()
train[["fico","default_24m"]].head()

,fico,default_24m
0,734,0
1,797,0
2,670,0
3,668,0
4,790,0


In [3]:
total_bad=train["default_24m"].sum()
total_good=len(train)-total_bad
print(total_bad,total_good)

776 170993


In [4]:
##cut credit score into 6 groups of equal size

train["fico_bin"]= pd.qcut(train["fico"], q=6, duplicates='drop')
train[["fico","fico_bin"]].head()

,fico,fico_bin
0,734,"(705.0, 737.0]"
1,797,"(781.0, 797.0]"
2,670,"(577.999, 705.0]"
3,668,"(577.999, 705.0]"
4,790,"(781.0, 797.0]"


In [5]:
train.head()

,loan_id,vintage_year,split,default_24m,fico,dti,ltv,cltv,orig_upb,log_orig_upb,...,orig_term,two_plus_borrowers,first_time_buyer,multi_unit,occupancy,loan_purpose,channel,property_type,state,fico_bin
0,F15Q10000025,2015,train,0,734,15.0,65.0,80.0,417000.0,12.940842,...,180,1,0,0,P,P,R,SF,CO,"(705.0, 737.0]"
1,F15Q10000031,2015,train,0,797,26.0,37.0,37.0,110000.0,11.608236,...,360,1,0,0,S,P,R,PU,NC,"(781.0, 797.0]"
2,F15Q10000084,2015,train,0,670,33.0,90.0,90.0,297000.0,12.601487,...,180,1,0,0,P,N,R,SF,MI,"(577.999, 705.0]"
3,F15Q10000157,2015,train,0,668,30.0,75.0,75.0,180000.0,12.100712,...,180,1,0,1,I,N,C,SF,UT,"(577.999, 705.0]"
4,F15Q10000167,2015,train,0,790,29.0,95.0,95.0,150000.0,11.918391,...,360,1,1,0,P,P,R,MH,WI,"(781.0, 797.0]"


In [6]:
# count loans and defaults in each group
table=train.groupby("fico_bin", observed=True)["default_24m"].agg(loans="count",defaults="sum")
table["non_defaults"]=table["loans"]-table["defaults"]
table["default_rate_pct"]=100*table["defaults"]/table["loans"]
table

,loans,defaults,non_defaults,default_rate_pct
fico_bin,,,,
"(577.999, 705.0]",28845,423,28422,1.466459
"(705.0, 737.0]",28705,172,28533,0.599199
"(737.0, 762.0]",29325,88,29237,0.300085
"(762.0, 781.0]",29091,46,29045,0.158125
"(781.0, 797.0]",29014,24,28990,0.082719
"(797.0, 839.0]",26782,23,26759,0.085879


In [7]:
# : compute the WoE Weight of Evidence. In credit risk charts, 
# it shows how strongly a group—such as a FICO score range—leans toward 
# non-default or default, compared with the overall dataset.
table["share_of_defaults"]=table["defaults"]/total_bad
table["share_of_non_defaults"]=table["non_defaults"]/total_good
table["WoE"]=np.log(table["share_of_defaults"]/table["share_of_non_defaults"])
table

,loans,defaults,non_defaults,default_rate_pct,share_of_defaults,share_of_non_defaults,WoE
fico_bin,,,,,,,
"(577.999, 705.0]",28845,423,28422,1.466459,0.545103,0.166217,1.187679
"(705.0, 737.0]",28705,172,28533,0.599199,0.221649,0.166866,0.283903
"(737.0, 762.0]",29325,88,29237,0.300085,0.113402,0.170984,-0.410628
"(762.0, 781.0]",29091,46,29045,0.158125,0.059278,0.169861,-1.052735
"(781.0, 797.0]",29014,24,28990,0.082719,0.030928,0.169539,-1.701427
"(797.0, 839.0]",26782,23,26759,0.085879,0.029639,0.156492,-1.663907


In [8]:
# compute the information value
iv= ((table["share_of_defaults"]-table["share_of_non_defaults"])*table["WoE"]).sum()
iv

np.float64(1.0525140559334543)

Set up the encoder:

WoEEncoder class automates binning and WoE calculation for all the numeric and categorical columns. To understand that part, add the script’s range_label function and full WoEEncoder class in a new notebook cell. The class creates a WoE table for each feature, then can use those tables to convert rows into WoE values.

A function lets us run the same steps on any feature. Smoothing adds half a loan to every bin, so a bin with zero defaults can’t make the math blow up.

In [9]:
def woe_table(bin, y ,smoothing=0.5):
    t=pd.DataFrame({"bin":bin,"y":y})
    g=t.groupby("bin")["y"].agg(loans="count",defaults="sum")
    k=len(g)
    bad=g["defaults"]
    good=g["loans"]-bad
    share_bad=(bad+smoothing)/(total_bad+smoothing*k)
    share_good=(good + smoothing)/(total_good + smoothing * k)
    woe=np.log(share_bad / share_good)
    iv=(share_bad - share_good) * woe
    return pd.DataFrame({"loans": g["loans"], "defaults": bad, "woe": woe, "iv": iv})



choose the features and learn the bin edges.

The edges come from the training loans only. Validation and test loans will later be placed into these same bins, because the model must never learn anything from them. The first and last edge become minus and plus infinity so that unusual values in other years still land in an outer bin.

In [10]:
numeric = ["fico", "dti", "ltv", "cltv", "orig_upb", "orig_rate", "mi_pct", "orig_term"]
categorical = ["occupancy", "loan_purpose", "channel", "property_type",
               "two_plus_borrowers", "first_time_buyer", "multi_unit"]

edges = {}
for col in numeric:
    _, e = pd.qcut(train[col], q=6, retbins=True, duplicates="drop")
    e[0],e[-1]=-np.inf,np.inf
    edges[col]=e
edges["ltv"]

array([-inf,  58.,  71.,  78.,  80.,  90.,  inf])

a function that puts any loans into bins. Missing values become their own bin, labelled nan.

In [11]:
def bin_feature(data,col):
    if col in numeric:
        return pd.cut(data[col], bins=edges[col]).astype(str)
    return data[col].astype(str)

bin_feature(train,"fico").head(15)

0     (705.0, 737.0]
1     (781.0, 797.0]
2      (-inf, 705.0]
3      (-inf, 705.0]
4     (781.0, 797.0]
5     (705.0, 737.0]
6     (762.0, 781.0]
7     (737.0, 762.0]
8     (762.0, 781.0]
9     (781.0, 797.0]
10    (705.0, 737.0]
11    (781.0, 797.0]
12    (737.0, 762.0]
13     (-inf, 705.0]
14     (-inf, 705.0]
Name: fico, dtype: str

learn the WoE of every bin, and rank the features by IV.

In [12]:
woe_maps = {}
iv_by_feature = {}
for col in numeric + categorical:
    tbl = woe_table(bin_feature(train, col), train["default_24m"])
    woe_maps[col] = tbl["woe"]
    iv_by_feature[col] = tbl["iv"].sum()

pd.Series(iv_by_feature).sort_values(ascending=False).round(3)

fico                  1.042
orig_rate             0.266
two_plus_borrowers    0.240
dti                   0.210
ltv                   0.125
cltv                  0.113
mi_pct                0.064
occupancy             0.057
loan_purpose          0.051
orig_term             0.050
first_time_buyer      0.042
orig_upb              0.033
property_type         0.012
channel               0.006
multi_unit            0.003
dtype: float64

The usual reading: below 0.02 is useless, 0.02 to 0.1 weak, 0.1 to 0.3 medium, above 0.3 strong.

### Part B: fit the model and test it

turn any set of loans into WoE numbers. Each feature value is replaced by its bin’s WoE. A bin never seen in training gets 0, meaning “average risk”.

In [13]:
def to_woe(data):
    out=pd.DataFrame(index=data.index)
    for col in numeric+categorical:
        out[col]=bin_feature(data,col).map(woe_maps[col]).fillna(0)
    return out


prepare the three datasets.

In [14]:
valid=df[df["split"]=="valid"]
test=df[df["split"]=="test"]

X_train, y_train = to_woe(train), train["default_24m"]
X_valid, y_valid = to_woe(valid), valid["default_24m"]
X_test, y_test = to_woe(test), test["default_24m"]
print(f"shape of X_train {X_train.shape}")
print(f"shape of X_valid {X_valid.shape}")
print(f"shape of X_test {X_test.shape}")
X_train.head()

shape of X_train (171769, 15)
shape of X_valid (48196, 15)
shape of X_test (49489, 15)


,fico,dti,ltv,cltv,orig_upb,orig_rate,mi_pct,orig_term,occupancy,loan_purpose,channel,property_type,two_plus_borrowers,first_time_buyer,multi_unit
0,0.282948,-1.080545,-0.159288,0.000282,-0.151355,-0.909926,-0.114583,-0.493633,0.068817,0.135873,-0.071208,0.048764,-0.606314,-0.105591,0.007019
1,-1.684666,-0.408857,-0.522907,-0.492323,0.342887,-0.478032,-0.114583,0.096442,-0.828559,0.135873,-0.071208,-0.055772,-0.606314,-0.105591,0.007019
2,1.185002,-0.032973,0.275642,0.229206,-0.130165,-0.909926,-0.114583,-0.493633,0.068817,-0.403060,-0.071208,0.048764,-0.606314,-0.105591,0.007019
3,1.185002,-0.408857,-0.335649,-0.272575,-0.133753,-0.478032,-0.114583,-0.493633,-0.831255,-0.403060,0.083957,0.048764,-0.606314,-0.105591,-0.412482
4,-1.684666,-0.408857,0.561735,0.543380,-0.025858,-0.061472,0.565395,0.096442,0.068817,0.135873,-0.071208,0.779106,-0.606314,0.398986,0.007019


## fit the logistic regression

In [15]:
from sklearn.linear_model import LogisticRegression
model=LogisticRegression(C=1.0, max_iter=1000)
model.fit(X_train,y_train)

pd.Series(model.coef_[0], index=X_train.columns).round(3)



fico                  0.944
dti                   0.546
ltv                   0.327
cltv                  0.248
orig_upb              0.276
orig_rate             0.454
mi_pct                0.170
orig_term             0.012
occupancy             0.356
loan_purpose          0.167
channel               0.047
property_type         0.142
two_plus_borrowers    0.992
first_time_buyer      0.058
multi_unit            0.031
dtype: float64

In [16]:
from sklearn.metrics import roc_auc_score

for name,X,y in [("train", X_train,y_train), ("valid",X_valid, y_valid),("test", X_test, y_test)]:
    p=model.predict_proba(X)[:, 1]
    print(name, "| AUC", round(roc_auc_score(y,p), 3),
          "| actual default %", round(100 * y.mean(), 2),
          "| predicted default %", round(100 * p.mean(), 2))

train | AUC 0.822 | actual default % 0.45 | predicted default % 0.44
valid | AUC 0.811 | actual default % 0.81 | predicted default % 0.6
test | AUC 0.789 | actual default % 0.72 | predicted default % 0.71


collect the predictions

In [17]:
preds = {
    "train": (y_train.to_numpy(), model.predict_proba(X_train)[:, 1]),
    "valid": (y_valid.to_numpy(), model.predict_proba(X_valid)[:, 1]),
    "test": (y_test.to_numpy(), model.predict_proba(X_test)[:, 1]),
}

#### KS. 
calculates the KS statistic, a measure of how well the model separates defaults from non-defaults.
Rank all loans by score. KS is the biggest gap between the share of defaults you’ve caught and the share of good loans you’ve flagged. 0 means no separation, 1 means perfect.

roc_curve(y, p) checks different probability cutoffs and returns:
- tpr: fraction of defaults correctly ranked above each cutoff
- fpr: fraction of non-defaults incorrectly ranked above each cutoff

tpr - fpr measures the separation at each cutoff.
.max() returns the largest separation.

In [18]:
from sklearn.metrics import roc_curve, roc_auc_score
def ks(y,p):
    fpr,tpr,_=roc_curve(y,p)
    return (tpr-fpr).max()

for name,(y,p) in preds.items():
    print(name, "| KS", round(ks(y, p), 3), "| AUC", round(roc_auc_score(y, p), 3))
    

train | KS 0.51 | AUC 0.822
valid | KS 0.495 | AUC 0.811
test | KS 0.457 | AUC 0.789


### decile table. 

Sort the test loans from riskiest to safest, cut them into ten equal groups, and see where the defaults fall. Decile 1 is the riskiest 10%.

In [19]:
def decile_table(y, p, n=10):
    t = pd.DataFrame({"y": y, "p": p})
    t["decile"] = pd.qcut(t["p"].rank(method="first"), n, labels=list(range(n, 0, -1))).astype(int)
    g = t.groupby("decile").agg(loans=("y", "size"), defaults=("y", "sum"), avg_pred=("p", "mean"))
    g["default_rate_pct"] = 100 * g["defaults"] / g["loans"]
    g["avg_pred_pct"] = 100 * g["avg_pred"]
    g["lift"] = g["default_rate_pct"] / (100 * t["y"].mean())
    g["cum_defaults_pct"] = 100 * g["defaults"].cumsum() / g["defaults"].sum()
    return g[["loans", "defaults", "default_rate_pct", "avg_pred_pct", "lift", "cum_defaults_pct"]].round(2)

decile_table(*preds["test"])

,loans,defaults,default_rate_pct,avg_pred_pct,lift,cum_defaults_pct
decile,,,,,,
1,4949,140,2.83,2.94,3.91,39.11
2,4949,80,1.62,1.47,2.23,61.45
3,4949,47,0.95,0.94,1.31,74.58
4,4949,34,0.69,0.62,0.95,84.08
5,4948,16,0.32,0.41,0.45,88.55
6,4949,16,0.32,0.28,0.45,93.02
7,4949,5,0.10,0.19,0.14,94.41
8,4949,10,0.20,0.14,0.28,97.21
9,4949,9,0.18,0.08,0.25,99.72


#### : confidence intervals.

Re-draw the test set with replacement 500 times and see how much AUC moves. This is the bootstrap, and it tells you how much of a number is skill and how much is luck.

In [20]:
rng = np.random.default_rng(42)

def bootstrap_ci(y, p, metric, n=500):
    vals = []
    for _ in range(n):
        idx = rng.integers(0, len(y), len(y))
        vals.append(metric(y[idx], p[idx]))
    return np.percentile(vals, [2.5, 97.5])

for name in ["valid", "test"]:
    y, p = preds[name]
    lo, hi = bootstrap_ci(y, p, roc_auc_score)
    print(name, "AUC 95% interval:", round(lo, 3), "to", round(hi, 3))

valid AUC 95% interval: 0.795 to 0.828
test AUC 95% interval: 0.769 to 0.81


### calibration, checked statistically.
For each year, compare the defaults that actually happened with the defaults the model expected. The z-score says how many standard deviations of pure chance separate them. Within about plus or minus 2 means the gap is consistent with noise.

In [21]:
rows = []
for name, (y, p) in preds.items():
    expected = p.sum()
    actual = y.sum()
    rows.append({
        "split": name,
        "loans": len(y),
        "actual_defaults": int(actual),
        "expected_defaults": round(expected, 1),
        "actual_to_expected": round(actual / expected, 2),
        "z_score": round((actual - expected) / np.sqrt(expected), 1),
    })
pd.DataFrame(rows)

,split,loans,actual_defaults,expected_defaults,actual_to_expected,z_score
0,train,171769,776,761.2,1.02,0.5
1,valid,48196,388,287.1,1.35,6.0
2,test,49489,358,352.3,1.02,0.3


### Feature ablation (step B)

This asks whether some features can go, and whether interest rate is acting as a stand-in for the year. We keep using the same notebook, and you type the cells yourself. We still don’t touch 2019, and we also don’t look at the 2020-2022 replay years for choosing features, since that’s our monitoring demo.

In [22]:
# a function that refits the model on a chosen set of features. 
# The WoE columns already exist, so dropping a feature is just dropping a column.

def fit_eval(features, C=1.0):
    m = LogisticRegression(C=C, max_iter=1000).fit(X_train[features], y_train)
    out = {}
    for name, X, y in [("valid", X_valid, y_valid), ("test", X_test, y_test)]:
        p = m.predict_proba(X[features])[:, 1]
        out[f"{name}_auc"] = round(roc_auc_score(y, p), 3)
        out[f"{name}_ks"] = round(ks(y, p), 3)
        out[f"{name}_actual_vs_expected"] = round(y.sum() / p.sum(), 2)
    return out

In [23]:
# compare feature sets.

all_features = list(X_train.columns)
weak = ["channel", "multi_unit", "property_type", "orig_term"]

sets = {
    "all 15 features": all_features,
    "drop weak four": [f for f in all_features if f not in weak],
    "drop orig_rate": [f for f in all_features if f != "orig_rate"],
    "drop orig_rate and orig_upb": [f for f in all_features if f not in ["orig_rate", "orig_upb"]],
    "drop cltv": [f for f in all_features if f != "cltv"],
    "drop weak four and orig_rate": [f for f in all_features if f not in weak + ["orig_rate"]],
}
pd.DataFrame({name: fit_eval(feats) for name, feats in sets.items()}).T

,valid_auc,valid_ks,valid_actual_vs_expected,test_auc,test_ks,test_actual_vs_expected
all 15 features,0.811,0.495,1.35,0.789,0.457,1.02
drop weak four,0.811,0.493,1.35,0.789,0.456,1.01
drop orig_rate,0.806,0.482,1.43,0.786,0.455,1.24
drop orig_rate and orig_upb,0.806,0.473,1.43,0.785,0.450,1.24
drop cltv,0.811,0.489,1.36,0.789,0.455,1.02
drop weak four and orig_rate,0.805,0.480,1.44,0.786,0.453,1.25


#### Does the interest rate effect hold within each year? 
If higher rates really signal riskier borrowers, defaults should rise with the rate inside every vintage. If the pattern only appears across years, rate is mostly standing in for the calendar.

In [24]:
t = train.copy()
t["rate_bin"] = pd.qcut(t["orig_rate"], 4, duplicates="drop")

print("Default rate % by vintage and rate quartile")
print(pd.crosstab(t["vintage_year"], t["rate_bin"], values=t["default_24m"], aggfunc="mean").mul(100).round(2))
print("\nLoans in each cell")
print(pd.crosstab(t["vintage_year"], t["rate_bin"]))

Default rate % by vintage and rate quartile
rate_bin      (2.249, 3.625]  (3.625, 4.0]  (4.0, 4.375]  (4.375, 6.0]
vintage_year                                                          
2013                    0.12          0.26          0.24          0.64
2014                    0.21          0.31          0.25          0.76
2015                    0.22          0.30          0.55          1.21
2016                    0.26          0.58          0.83          1.20

Loans in each cell
rate_bin      (2.249, 3.625]  (3.625, 4.0]  (4.0, 4.375]  (4.375, 6.0]
vintage_year                                                          
2013                   15268          6130          5753          8467
2014                    5765          4174         15101         17653
2015                   10299         15223         14913          5548
2016                   18597         16855          8360          3663


### Lock the features and test calibration

In [25]:
# ten-feature model against the fifteen-feature one.

final_features = ["fico", "dti", "ltv", "orig_upb", "orig_rate", "mi_pct",
                  "occupancy", "loan_purpose", "two_plus_borrowers", "first_time_buyer"]
fit_eval(final_features)

{'valid_auc': 0.811,
 'valid_ks': np.float64(0.488),
 'valid_actual_vs_expected': np.float64(1.35),
 'test_auc': 0.788,
 'test_ks': np.float64(0.453),
 'test_actual_vs_expected': np.float64(1.01)}

In [27]:
# fit the final model and score 2017 and 2018.

final = LogisticRegression(C=1.0, max_iter=1000).fit(X_train[final_features],y_train)

def final_p(X):
    return final.predict_proba(X[final_features])[:,1]
p_valid, p_test = final_p(X_valid),final_p(X_test)

In [28]:
# find the single shift.

from scipy.optimize import brentq
from scipy.special import logit,expit

pooled_y = np.concatenate([y_valid.to_numpy(),y_test.to_numpy()])
pooled_p= np.concatenate([p_valid,p_test])

def gap(a):
    return expit(logit(pooled_p) + a).mean() - pooled_y.mean()

shift = brentq(gap, -2, 2)
print("log-odds shift:", round(shift, 3), "| odds multiplier:", round(np.exp(shift), 3))


log-odds shift: 0.157 | odds multiplier: 1.17


In [29]:
## effect on 2017 and 2018. These are the years we fitted on, so this only shows the trade-off between them, not a test.

def calibrated(p):
    return expit(logit(p) + shift)

for name, y, p in [("valid 2017", y_valid.to_numpy(),  p_valid), ("test 2018", y_test.to_numpy(), p_test)]:
    for label, q in [("before", p), ("after", calibrated(p))]:
        print(name, label, "| actual/expected", round(y.sum() / q.sum(), 2))


valid 2017 before | actual/expected 1.35
valid 2017 after | actual/expected 1.16
test 2018 before | actual/expected 1.01
test 2018 after | actual/expected 0.87


In [30]:
## the real test on 2019, run once.

spare=df[df["split"]=="spare"]
X_spare, y_spare= to_woe(spare), spare["default_24m"].to_numpy()
p_spare = final_p(X_spare)

for label, q in [("before", p_spare), ("after", calibrated(p_spare))]:
    expected = q.sum()
    print("2019", label,
          "| actual/expected", round(y_spare.sum() / expected, 2),
          "| z", round((y_spare.sum() - expected) / np.sqrt(expected), 1),
          "| AUC", round(roc_auc_score(y_spare, q), 3))

2019 before | actual/expected 1.17 | z 2.8 | AUC 0.728
2019 after | actual/expected 1.0 | z 0.0 | AUC 0.728


### AUC fall for 2019: compared to 2017, 2018

Diagnose the drop

This step uses two tools, and neither lets us tune the model on 2019.
-  PSI: Population Stability Index: measures how much each feature’s distribution moved compared with training. It doesn’t use labels at all. Below 0.1 is stable, 0.1 to 0.25 is a moderate shift, and above 0.25 is a major shift. This is the same measure the monitoring layer will use in Week 6.
- Single Feature AUC by year: r shows how well each feature alone ranks borrowers in each year, so we can see where the ranking weakened.

In [31]:
## the PSI function: It reuses the bins that we already created and 
# compares the share of loans in each bin against training

def psi(ref, cur, col, eps=1e-6):
    r=bin_feature(ref,col).value_counts(normalize=True)
    c=bin_feature(cur,col).value_counts(normalize=True)
    bins=r.index.union(c.index)
    r=r.reindex(bins).fillna(0)+eps
    c=c.reindex(bins).fillna(0)+eps
    return float(((c - r) * np.log(c/r)).sum())

In [33]:
## PSI of each feature in year 2017,2018,2019 against training
periods={"2017":valid, "2018": test,"2019":spare}
pd.DataFrame(
    {name:{f: psi(train,d,f) for f in final_features} for name,d in periods.items()}
).round(3)

,2017,2018,2019
fico,0.022,0.018,0.008
dti,0.029,0.063,0.037
ltv,0.019,0.045,0.033
orig_upb,0.003,0.003,0.042
orig_rate,0.271,2.014,0.247
mi_pct,0.000,0.000,0.000
occupancy,0.005,0.002,0.001
loan_purpose,0.080,0.221,0.021
two_plus_borrowers,0.008,0.015,0.015
first_time_buyer,0.018,0.059,0.019


In [34]:
# How well each feature alone ranks, by year. Each column is WoE, 
# where higher means riskier, so a feature with signal gives an AUC above 0.5.

X_by = {"train": X_train, "2017": X_valid, "2018": X_test, "2019": X_spare}
y_by = {"train": y_train.to_numpy(), "2017": y_valid.to_numpy(),
        "2018": y_test.to_numpy(), "2019": y_spare}

pd.DataFrame(
    {name: {f: roc_auc_score(y_by[name], X_by[name][f]) for f in final_features}
     for name in X_by}
).round(3)

,train,2017,2018,2019
fico,0.766,0.738,0.734,0.688
dti,0.615,0.612,0.595,0.589
ltv,0.599,0.625,0.599,0.560
orig_upb,0.545,0.507,0.536,0.491
orig_rate,0.642,0.659,0.585,0.577
mi_pct,0.547,0.553,0.535,0.509
occupancy,0.532,0.537,0.526,0.516
loan_purpose,0.550,0.523,0.498,0.507
two_plus_borrowers,0.619,0.619,0.609,0.589
first_time_buyer,0.541,0.568,0.560,0.545


##### Replace the rate level with a rate spread
A rate spread is the loan’s rate minus the average rate of loans of the same type (long or short term) in the same month. It removes the market level, so it should stay stable across years while keeping the borrower-specific pricing signal.

In [37]:
# add the origination month and compute the spread.
fp = duckdb.sql(
    "SELECT loan_id, first_payment_date "
    "FROM read_parquet('../data/silver/orig/*/*.parquet', hive_partitioning=true)"
).df()
df=df.merge(fp, on="loan_id", how="left")
df["term_long"] = df["orig_term"]>=300
df["month_rate"]=df.groupby(["first_payment_date", "term_long"])["orig_rate"].transform("mean")
df["rate_spread"]=df["orig_rate"]-df["month_rate"]
df[["orig_rate", "month_rate", "rate_spread"]].describe().round(2)




,orig_rate,month_rate,rate_spread
count,469424.00,469424.00,469424.00
mean,4.03,4.03,0.00
std,0.88,0.79,0.38
min,1.62,2.47,-3.53
25%,3.38,3.46,-0.24
50%,4.00,4.04,-0.05
75%,4.50,4.46,0.19
max,8.12,7.00,2.70


In [40]:
## rebuild the splits and learn the WoE of the new feature.
train=df[df["split"]=="train"].copy()
valid = df[df["split"] == "valid"].copy()
test = df[df["split"] == "test"].copy()
spare = df[df["split"] == "spare"].copy()

if "rate_spread" not in numeric:
    numeric.append("rate_spread")

_, e=pd.qcut(train["rate_spread"], q=6, retbins=True, duplicates="drop")
e[0], e[-1] = -np.inf, np.inf
edges["rate_spread"] = e

tbl = woe_table(bin_feature(train, "rate_spread"), train["default_24m"])
woe_maps["rate_spread"] = tbl["woe"]
tbl

,loans,defaults,woe,iv
bin,,,,
"(-0.045, 0.0875]",28480,128,-0.005157,0.000004
"(-0.153, -0.045]",28573,72,-0.582741,0.042896
"(-0.278, -0.153]",28898,78,-0.514359,0.034861
"(-inf, -0.278]",28676,46,-1.031391,0.111124
"(0.0875, 0.289]",28520,195,0.415427,0.035441
"(0.289, inf]",28622,257,0.689475,0.113535


In [41]:
## rebuild the model inputs.

X_train, y_train = to_woe(train), train["default_24m"]
X_valid, y_valid = to_woe(valid), valid["default_24m"]
X_test, y_test = to_woe(test), test["default_24m"]
X_spare, y_spare = to_woe(spare), spare["default_24m"].to_numpy()

In [42]:
##compare the options on 2017 and 2018.
base = ["fico", "dti", "ltv", "orig_upb", "mi_pct", "occupancy",
        "loan_purpose", "two_plus_borrowers", "first_time_buyer"]
sets = {
    "current: with orig_rate": base + ["orig_rate"],
    "no rate at all": base,
    "rate_spread instead of orig_rate": base + ["rate_spread"],
    "both": base + ["orig_rate", "rate_spread"],
}
pd.DataFrame({name: fit_eval(feats) for name, feats in sets.items()}).T

,valid_auc,valid_ks,valid_actual_vs_expected,test_auc,test_ks,test_actual_vs_expected
current: with orig_rate,0.811,0.488,1.35,0.788,0.453,1.01
no rate at all,0.805,0.475,1.44,0.785,0.452,1.26
rate_spread instead of orig_rate,0.814,0.498,1.46,0.794,0.469,1.28
both,0.814,0.496,1.41,0.794,0.464,1.14


In [43]:
## how stable is the new feature? (No default labels used.)
for name, d in {"2017": valid, "2018": test, "2019": spare}.items():
    print(name, "PSI of rate_spread:", round(psi(train, d, "rate_spread"), 3))

2017 PSI of rate_spread: 0.024
2018 PSI of rate_spread: 0.028
2019 PSI of rate_spread: 0.069


Decision: adopt rate_spread and drop orig_rate

Two caveats for the model card:

- The spread is the lender’s own pricing, so it can’t be used to approve or decline an application, since the price is set afterward. This model scores loans that are already priced: portfolio risk, early warning, pricing adequacy. For approval-time use, the no-rate version loses only about 0.009 AUC, and we’ll document it as a variant. This reframes the threshold step: it becomes a review or watchlist cutoff for booked loans.
- In production, the monthly market rate has to come from outside the data. Here we approximate it from the dataset’s own loans (about 4,000 per month in the sample). In production you’d feed it from a market rate series, which is one more input the pipeline must supply.

### Lock scorecard v1.1 and recalibrate

We’ll refit on the final ten features, recompute the shift, and check the 2017 and 2018 calibration. We still leave 2019 alone. It has been used for diagnosis, and any number we report from it carries that label.

In [44]:
final_features=base+["rate_spread"]
final = LogisticRegression(C=1.0, max_iter=1000).fit(X_train[final_features],y_train)

def final_p(X):
    return final.predict_proba(X[final_features])[:, 1]
p_valid, p_test=final_p(X_valid), final_p(X_test)
pd.Series(final.coef_[0], index=final_features).round(3)

fico                  0.898
dti                   0.553
ltv                   0.508
orig_upb              0.229
mi_pct                0.273
occupancy             0.410
loan_purpose          0.231
two_plus_borrowers    0.989
first_time_buyer      0.124
rate_spread           0.499
dtype: float64

the single log-odds shift, fitted on pooled 2017 and 2018.

In [46]:
pooled_y=np.concatenate([y_valid.to_numpy(), y_test.to_numpy()])
pooled_p = np.concatenate([p_valid, p_test])

shift = brentq(lambda a: expit(logit(pooled_p) + a).mean() - pooled_y.mean(), -2, 2)
print("log-odds shift:", round(shift, 3), "| odds multiplier:", round(np.exp(shift), 3))

log-odds shift: 0.317 | odds multiplier: 1.374


before and after, by year.

In [47]:
def calibrated(p):
    return expit(logit(p) + shift)

for name, y, p in [("2017", y_valid.to_numpy(), p_valid), ("2018", y_test.to_numpy(), p_test)]:
    for label, q in [("before", p), ("after", calibrated(p))]:
        expected = q.sum()
        print(name, label,
              "| actual/expected", round(y.sum() / expected, 2),
              "| z", round((y.sum() - expected) / np.sqrt(expected), 1))

2017 before | actual/expected 1.46 | z 7.5
2017 after | actual/expected 1.07 | z 1.3
2018 before | actual/expected 1.28 | z 4.6
2018 after | actual/expected 0.93 | z -1.3


In [48]:
decile_table(y_test.to_numpy(), calibrated(p_test))

,loans,defaults,default_rate_pct,avg_pred_pct,lift,cum_defaults_pct
decile,,,,,,
1,4949,145,2.93,3.40,4.05,40.50
2,4949,83,1.68,1.59,2.32,63.69
3,4949,38,0.77,0.99,1.06,74.30
4,4949,33,0.67,0.63,0.92,83.52
5,4948,23,0.46,0.41,0.64,89.94
6,4949,8,0.16,0.28,0.22,92.18
7,4949,10,0.20,0.19,0.28,94.97
8,4949,11,0.22,0.13,0.31,98.04
9,4949,5,0.10,0.08,0.14,99.44
